# Fase 2 · Do protótipo ao agente: ferramentas, raciocínio ReAct e prompts avançados

**Projeto:** Assistente de Viagens com LangChain Agents
**Autora:** Shaiane Alves Tonin

Nesta fase, o MVP da Fase 1 (uma chain que só transforma texto em JSON) vira um **agente**: um sistema que decide, passo a passo, se responde direto ou se age sobre o mundo por meio de uma ferramenta.

| Requisito do enunciado | Seção |
|---|---|
| Comparação de orquestradores e justificativa do LangChain | 1 |
| Ferramentas reais, incluindo API externa via *function calling* | 2, 3 |
| Prompt PTCF com *few-shot* e *chain-of-thought* | 4 |
| ReAct com `initialize_agent` + `ZERO_SHOT_REACT_DESCRIPTION` | 5, 6 |
| Memória da chain configurada | 5, 7 |
| Chains sequenciais e paralelas | 8 |
| Roteamento dinâmico com `RouterChain` | 9 |
| Output parsers com tratamento de erros | 10 |
| FAISS como ferramenta de busca semântica | 2 |
| Callbacks para logging e depuração | 11 |
| Justificativa das escolhas de confiabilidade | 12 |

In [1]:
# Setup: funciona no VS Code (venv local) e no Google Colab.
import os, sys, json, time, warnings
from pathlib import Path
from datetime import date, timedelta

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    os.chdir("/content/projeto-langchain-agentes")
    %pip install -q -r requirements.txt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

# initialize_agent, RouterChain e as memórias "Conversation*" são APIs legadas exigidas pelo enunciado;
# o LangChain 0.3 emite avisos de depreciação para elas. Silenciamos só esses avisos.
from langchain_core._api import LangChainDeprecationWarning
warnings.filterwarnings("ignore", category=LangChainDeprecationWarning)

from viagens.config import USE_DUMMY, get_llm
from viagens.callbacks import LogExecucaoHandler

llm = get_llm(temperature=0)
log_handler = LogExecucaoHandler(sessao="fase2")
CFG = {"callbacks": [log_handler]}
print("Modo dummy:", USE_DUMMY, "| modelo:", getattr(llm, "model_name", None))

Modo dummy: False | modelo: gpt-4o-mini


## 1. Escolha do orquestrador

Critérios objetivos:
- **Autonomia**: quanto o framework decide sozinho o próximo passo, desde um fluxo fixo até um agente que planeja e delega.
- **Abstração**: quanto o framework esconde (construtos prontos de alto nível ou peças de baixo nível).
- **Modularidade**: facilidade de trocar LLM, ferramentas, memória e vector store sem reescrever o resto.

| Orquestrador | Autonomia | Abstração | Modularidade | Observação para este projeto |
|---|---|---|---|---|
| **LangChain** (`AgentExecutor`, chains, LCEL) | Média: ciclo ReAct/tool-calling com limite de iterações | Alta, com construtos prontos (`initialize_agent`, `RouterChain`, memórias, parsers) e LCEL por baixo | **Alta**: interface `Runnable` única para LLMs, tools, retrievers e parsers, com centenas de integrações (OpenAI, FAISS, Open-Meteo via tools) | Cobre **todos** os construtos exigidos e é a base da Fase 1 |
| **LangGraph** | Alta: grafos com ciclos, estado persistente e *human-in-the-loop* | Baixa/média: o desenvolvedor modela nós, arestas e estado | Alta (reusa os componentes do LangChain) | Ideal para **orquestrar vários agentes**, que é o tema da próxima disciplina |
| **CrewAI** | Alta: agentes com papéis que colaboram em tarefas | Muito alta: "crew", "role", "task" | Média: menos controle fino sobre prompt e parsing | Pensado para multiagente; excesso de abstração para um agente único auditável |
| **AutoGen (AG2)** | Alta: agentes conversando entre si | Média | Média | Foco em conversas multiagente e execução de código |
| **LlamaIndex Agents** | Média | Alta para RAG | Média/alta | Muito forte em indexação e RAG, mais fraco em chains, roteamento e memória conversacional |

**Por que o LangChain nesta etapa:**
1. **Continuidade do MVP.** A chain da Fase 1 já é um `Runnable` LCEL. Ela entra como peça do agente sem reescrita (ver seção 8).
2. **Autonomia na medida certa.** Para um agente único que precisa ser **avaliado por trajetória** (Fase 5), queremos um ciclo controlado (`max_iterations`, `early_stopping_method`, `handle_parsing_errors`), não colaboração aberta entre agentes.
3. **Modularidade comprovada aqui mesmo.** O mesmo agente roda com `gpt-4o-mini` ou com o `DummyChatModel`, e com embeddings OpenAI ou o embedding lexical, trocando **uma linha** (`get_llm`, `get_embeddings`).
4. **Caminho de evolução.** Na próxima disciplina, o agente vira um nó de um grafo LangGraph, reaproveitando ferramentas, prompts e memória.

## 2. As ferramentas do agente

Toda a lógica está em `viagens/tools.py`. Cada ferramenta tem uma **descrição pensada para o modelo**, porque é lendo esse texto que o agente decide quando usá-la.

| Ferramenta | Tipo | Fonte |
|---|---|---|
| `previsao_tempo` | **API externa real** | Open-Meteo (geocoding + forecast/archive), sem chave |
| `converter_moeda` | **API externa real** | Frankfurter (cotações do Banco Central Europeu), sem chave |
| `buscar_regras_viagem` | Busca semântica | Índice **FAISS** sobre 14 documentos (regras de entrada, bagagem, saúde, IOF, guias) |
| `calculadora` | Função fixa (hardcoded) | Avaliação aritmética segura via `ast`, sem `eval` |

Antes de entregar as ferramentas ao agente, cada uma é testada isoladamente, chamando as APIs reais.

In [2]:
from viagens.tools import ferramentas_react, ferramentas_estruturadas

tools_react = ferramentas_react()
tools_fc = ferramentas_estruturadas()
por_nome = {t.name: t for t in tools_fc}

import pandas as pd
pd.set_option("display.max_colwidth", 120)
pd.DataFrame([{"nome": t.name, "descrição (lida pelo LLM)": t.description[:150] + "..."} for t in tools_react])

,nome,descrição (lida pelo LLM)
0,previsao_tempo,"Consulta a previsão do tempo REAL (API Open-Meteo) de uma cidade: temperaturas máx./mín., chuva e condição por dia. ..."
1,converter_moeda,"Converte um valor entre moedas com a cotação REAL do dia (API Frankfurter, Banco Central Europeu). Use códigos ISO: ..."
2,buscar_regras_viagem,"Busca semântica na base de conhecimento de viagens (índice FAISS): regras de entrada e visto, seguro-viagem, bagagem..."
3,calculadora,"Calculadora aritmética exata. Use para somar custos, dividir orçamentos, porcentagens. Entrada: a expressão, ex.: '2..."


In [3]:
amanha = (date.today() + timedelta(days=1)).isoformat()
print("previsao_tempo →", por_nome["previsao_tempo"].invoke({"cidade": "Lisboa", "data_inicio": amanha, "dias": 2}, config=CFG)[:400], "\n")
print("converter_moeda →", por_nome["converter_moeda"].invoke({"valor": 1000, "de": "BRL", "para": "EUR"}, config=CFG), "\n")
print("calculadora →", por_nome["calculadora"].invoke({"expressao": "15000 * 0.035"}, config=CFG), "\n")
# erro tratado: a Frankfurter não cota o peso argentino -> a ferramenta devolve a mensagem em vez de quebrar
print("converter_moeda (erro tratado) →", por_nome["converter_moeda"].invoke({"valor": 500, "de": "BRL", "para": "ARS"}, config=CFG))

previsao_tempo → {"cidade": "Lisboa", "pais": "Portugal", "tipo": "previsao", "fonte": "Open-Meteo", "periodo_consultado": ["2026-11-13", "2026-11-14"], "dias": [{"data": "2026-11-13", "temp_max": 19.7, "temp_min": 15.7, "chuva_mm": 0.4, "condicao": "nublado", "prob_chuva_%": 53}, {"data": "2026-11-14", "temp_max": 19.3, "temp_min": 14.8, "chuva_mm": 0.0, "condicao": "nublado", "prob_chuva_%": 43}]} 



converter_moeda → {"valor": 1000.0, "de": "BRL", "para": "EUR", "convertido": 160.67, "taxa": 0.16067, "data_cotacao": "2026-11-12", "fonte": "Frankfurter (BCE)"} 

calculadora → 525.0 



converter_moeda (erro tratado) → Erro em converter_moeda: Moeda não suportada pela Frankfurter: BRL->ARS (a API só cobre moedas cotadas pelo Banco Central Europeu; ARS e CLP, por exemplo, não).


### FAISS como ferramenta de busca semântica

O índice é construído **uma vez** a partir de `data/docs/*.md` (chunks de 700 caracteres, com sobreposição de 100) e salvo em `data/faiss_index/`. A Fase 4 vai **carregar este mesmo índice** sem reconstruí-lo.

- Embeddings no modo real: `text-embedding-3-small` (OpenAI).
- Embeddings no modo dummy: hashing lexical determinístico, sem chave.

In [4]:
from viagens.conhecimento import construir_indice, carregar_documentos, dividir, INDEX_DIR

if not (INDEX_DIR / "index.faiss").exists():
    construir_indice()
docs = carregar_documentos()
print(f"{len(docs)} documentos → {len(dividir(docs))} chunks | índice em {INDEX_DIR.relative_to(ROOT)}")

for q in ["Preciso de seguro-viagem para ir a Portugal?", "Posso levar power bank na mala despachada?"]:
    trechos = json.loads(por_nome["buscar_regras_viagem"].invoke({"consulta": q}))
    print(f"\n{q}")
    for t in trechos:
        print(f"   {t['distancia']:.3f}  {t['fonte']:<22} {t['trecho'][:90]!r}")

14 documentos → 31 chunks | índice em data/faiss_index/openai



Preciso de seguro-viagem para ir a Portugal?
   0.682  schengen_regras.md     '## Seguro-viagem obrigatório\nÉ exigido seguro-viagem com cobertura mínima de 30.000 euros '
   0.839  saude_seguro.md        '# Saúde e vacinas para viagens internacionais\n\n## Febre amarela\nAlguns países exigem o Cer'
   0.908  portugal_entrada.md    '## Documentos exigidos na fronteira\n- Passaporte válido por pelo menos 3 meses após a data'



Posso levar power bank na mala despachada?
   0.808  bagagem_aerosul.md     '## Bagagem de mão\n- 1 volume de até 10 kg nas dimensões 55 x 35 x 25 cm, mais 1 item pesso'
   1.106  saude_seguro.md        '## Medicamentos\nLeve medicamentos de uso contínuo na bagagem de mão, na embalagem original'
   1.132  bagagem_aerosul.md     '# Política de bagagem da AeroSul Linhas Aéreas\n\nA AeroSul é uma companhia fictícia usada n'


## 3. API externa via *function calling*

No *function calling*, o modelo não escreve texto livre para chamar a ferramenta: ele devolve uma chamada **estruturada** (nome e argumentos JSON validados pelo schema Pydantic da ferramenta). O código executa a chamada e devolve o resultado como `ToolMessage`. Abaixo, o ciclo completo com as duas APIs reais.

In [5]:
from langchain_core.utils.function_calling import convert_to_openai_tool
print("Schema enviado ao modelo para converter_moeda:")
print(json.dumps(convert_to_openai_tool(por_nome["converter_moeda"]), ensure_ascii=False, indent=1))

Schema enviado ao modelo para converter_moeda:
{
 "type": "function",
 "function": {
  "name": "converter_moeda",
  "description": "Converte um valor entre moedas com a cotação REAL do dia (API Frankfurter, Banco Central Europeu). Use códigos ISO: BRL, EUR, USD, GBP, JPY, CHF... Não cobre ARS nem CLP.",
  "parameters": {
   "properties": {
    "valor": {
     "description": "Valor a converter",
     "type": "number"
    },
    "de": {
     "description": "Moeda de origem, código ISO de 3 letras (ex.: BRL)",
     "type": "string"
    },
    "para": {
     "description": "Moeda de destino, código ISO de 3 letras (ex.: EUR)",
     "type": "string"
    }
   },
   "required": [
    "valor",
    "de",
    "para"
   ],
   "type": "object"
  }
 }
}


In [6]:
from langchain_core.messages import HumanMessage, SystemMessage

llm_fc = llm.bind_tools(tools_fc)
msgs = [SystemMessage(content=f"Você é um assistente de viagens. Hoje é {date.today()}. "
                              "Use as ferramentas para todo dado que muda com o tempo."),
        HumanMessage(content="Quanto ficam 2.500 reais em euros e como estará o tempo em Lisboa amanhã?")]

ai = llm_fc.invoke(msgs, config=CFG)
print("1) O modelo pediu", len(ai.tool_calls), "chamada(s) de ferramenta:")
for tc in ai.tool_calls:
    print("   ", tc["name"], tc["args"])

msgs.append(ai)
for tc in ai.tool_calls:
    tool_msg = por_nome[tc["name"]].invoke(tc, config=CFG)   # executa e devolve ToolMessage
    msgs.append(tool_msg)
    print(f"2) {tc['name']} executada → {tool_msg.content[:150]}...")

final = llm_fc.invoke(msgs, config=CFG)
print("\n3) Resposta final do modelo com os dados reais:\n", final.content)

1) O modelo pediu 2 chamada(s) de ferramenta:
    converter_moeda {'valor': 2500, 'de': 'BRL', 'para': 'EUR'}
    previsao_tempo {'cidade': 'Lisboa', 'data_inicio': '2026-11-13', 'dias': 1}


2) converter_moeda executada → {"valor": 2500.0, "de": "BRL", "para": "EUR", "convertido": 401.67, "taxa": 0.160668, "data_cotacao": "2026-11-12", "fonte": "Frankfurter (BCE)"}...


2) previsao_tempo executada → {"cidade": "Lisboa", "pais": "Portugal", "tipo": "previsao", "fonte": "Open-Meteo", "periodo_consultado": ["2026-11-13", "2026-11-13"], "dias": [{"dat...



3) Resposta final do modelo com os dados reais:
 A conversão de 2.500 reais para euros resulta em aproximadamente **401,67 euros**.

Quanto ao tempo em Lisboa amanhã (13 de novembro de 2026), a previsão é de:

- **Temperatura máxima:** 19,7°C
- **Temperatura mínima:** 15,7°C
- **Condição:** Nublado
- **Probabilidade de chuva:** 53%
- **Precipitação esperada:** 0,4 mm

Se precisar de mais informações, estou à disposição!


## 4. Prompt de sistema PTCF com *few-shot* e *chain-of-thought*

- **P**ersona: quem é o agente e qual a sua postura (cuidadoso com fatos, sem inventar dados).
- **T**arefa: o que ele faz, incluindo decompor pedidos compostos em etapas.
- **C**ontexto: a data de hoje, a moeda do usuário, os limites das ferramentas e **como tratar os casos ambíguos do domínio** ("Porto", "Santiago", datas fora do horizonte de previsão).
- **F**ormato: o raciocínio explícito no `Thought` e uma `Final Answer` curta que cita as fontes.

O **few-shot** traz três exemplos que cobrem justamente os casos difíceis: uma cidade ambígua, uma tarefa de duas etapas com dependência (converter e depois dividir) e uma mensagem que **não** precisa de ferramenta. O **chain-of-thought** aparece no campo `Thought` de cada exemplo, que mostra o raciocínio antes da ação. Isso torna o raciocínio depurável: dá para ler nos logs *por que* o agente escolheu cada ferramenta.

In [7]:
PREFIXO_PTCF = """### PERSONA
Você é a Viaja+, agente de viagens sênior que atende viajantes brasileiros. Você é cuidadosa com fatos: nunca inventa previsões, cotações, preços ou regras de entrada. Todo dado que muda com o tempo vem de uma ferramenta.

### TAREFA
Responder perguntas e ajudar a planejar viagens: clima no destino, câmbio, regras de entrada, documentos, bagagem e dicas. Quando o pedido tiver várias partes, decomponha em etapas e resolva uma etapa por ação.

### CONTEXTO
- Hoje é {hoje}. Calcule datas relativas ("amanhã", "fim de semana", "sábado") a partir disso. A moeda do usuário é o real (BRL).
- A previsão do tempo só cobre 16 dias; para datas mais distantes a ferramenta devolve referência histórica, e você deve dizer isso.
- Cidades ambíguas: "Porto" pode ser Porto (Portugal) ou Porto Alegre; "Santiago" pode ser Santiago do Chile ou Santiago de Compostela. Use a conversa anterior para decidir; se não der, assuma a opção mais provável e declare a suposição.
- Se uma ferramenta devolver erro, explique a limitação ao usuário e, se possível, use outra ferramenta.

### FORMATO
- Antes de cada ação, raciocine passo a passo no Thought: o que já sei, o que falta e qual ferramenta resolve a próxima etapa.
- Final Answer em português, com no máximo 6 frases, citando as fontes entre colchetes: [Open-Meteo], [Frankfurter], [base: nome_do_arquivo.md].
- Toda resposta ao usuário começa com "Final Answer:", MESMO quando nenhuma ferramenta for usada.

### EXEMPLOS
Question: Vai fazer frio no Porto no sábado?
Thought: "Porto" é ambíguo e não há conversa anterior. Para um viajante brasileiro perguntando sobre frio, o mais provável é Porto, Portugal. Vou assumir isso e avisar. Preciso da previsão.
Action: previsao_tempo
Action Input: Porto, 2026-11-14
Observation: cidade=Porto, pais=Portugal, 2026-11-14: 9 a 15 °C, chuva fraca
Thought: Tenho a previsão. Mínima de 9 °C é frio para um brasileiro.
Final Answer: Considerando Porto, em Portugal (se for Porto Alegre, me avise): no sábado a previsão é de 9 a 15 °C com chuva fraca; leve casaco e guarda-chuva [Open-Meteo].

Question: Tenho 5000 reais para 5 dias em Paris. Quanto dá por dia em euros?
Thought: São duas etapas dependentes: (1) converter 5000 BRL para EUR com a cotação do dia; (2) dividir o resultado por 5.
Action: converter_moeda
Action Input: 5000 BRL EUR
Observation: valor=5000 BRL, convertido=805.00 EUR, taxa=0.161
Thought: Etapa 1 concluída: 805.00 EUR. Agora a etapa 2, com a calculadora para não errar a conta.
Action: calculadora
Action Input: 805.00 / 5
Observation: 161.0
Thought: Concluí as duas etapas.
Final Answer: Seus R$ 5.000 equivalem a cerca de 805 euros hoje, aproximadamente 161 euros por dia [Frankfurter]. Lembre-se de que o IOF de 3,5% sobre gastos no exterior reduz esse valor.

Question: Obrigada pela ajuda!
Thought: É apenas um agradecimento; não preciso de ferramenta.
Final Answer: Por nada! Boa viagem e conte comigo para o que precisar.

(Os valores dos exemplos são ilustrativos; nas perguntas reais, use sempre as ferramentas.)

### FERRAMENTAS
Você tem acesso às seguintes ferramentas:"""
DIAS_SEMANA = ["segunda-feira", "terça-feira", "quarta-feira", "quinta-feira", "sexta-feira", "sábado", "domingo"]
PREFIXO_PTCF = PREFIXO_PTCF.replace("{hoje}", f"{DIAS_SEMANA[date.today().weekday()]}, {date.today().isoformat()}")

SUFIXO = """Comece! Raciocine no Thought antes de agir.

Histórico da conversa:
{chat_history}

Question: {input}
Thought:{agent_scratchpad}"""

MSG_ERRO_PARSING = ("Formato inválido. Responda com 'Thought:' seguido de 'Action:' e 'Action Input:' "
                    "OU de 'Final Answer:'. Não escreva os dois ao mesmo tempo.")

## 5. Memória da chain e agente ReAct (`initialize_agent` + `ZERO_SHOT_REACT_DESCRIPTION`)

**Escolha da memória: `ConversationSummaryBufferMemory`.** Uma conversa de planejamento de viagem tem tipicamente de 10 a 30 turnos, e as preferências ditas no início ("viajo com criança", "orçamento de 15 mil") continuam valendo no fim.

| Opção | Vantagem | Problema neste domínio |
|---|---|---|
| `ConversationBufferMemory` | Guarda tudo, literal | O prompt cresce sem limite: custo e latência sobem a cada turno |
| `ConversationBufferWindowMemory` (k turnos) | Tamanho fixo | **Esquece** as preferências ditas no começo da conversa |
| `ConversationTokenBufferMemory` | Limite por tokens | Também descarta o que é antigo, só que medido em tokens |
| **`ConversationSummaryBufferMemory`** | Turnos recentes **literais** + os antigos **resumidos** pelo LLM, com limite de tokens | Custa uma chamada extra de resumo quando o limite estoura, um custo aceitável |

O limite de produção é `max_token_limit=800`. Para a demonstração da seção 7 uso 250 tokens, para ver o resumo acontecer em poucos turnos. A memória de longo prazo, entre sessões, fica para a Fase 3.

In [8]:
from langchain.agents import AgentType, initialize_agent
from langchain.memory import ConversationSummaryBufferMemory
from langchain_core.prompts import PromptTemplate


PROMPT_RESUMO = PromptTemplate.from_template(
    "Resuma progressivamente a conversa entre o viajante (Human) e o assistente (AI), acrescentando as novas "
    "linhas ao resumo atual. Preserve SEMPRE: destino, datas, número de viajantes, orçamento, preferências "
    "e restrições do viajante, e conclusões já obtidas (valores convertidos, regras). Escreva em português "
    "e responda APENAS com o texto do novo resumo, sem títulos.\n\n"
    "Resumo atual:\n{summary}\n\nNovas linhas da conversa:\n{new_lines}\n\nNovo resumo:")


def nova_memoria(limite_tokens: int = 800):
    return ConversationSummaryBufferMemory(
        llm=get_llm(temperature=0), max_token_limit=limite_tokens, prompt=PROMPT_RESUMO,
        memory_key="chat_history", input_key="input", output_key="output")


def criar_agente_react(memoria):
    return initialize_agent(
        tools=tools_react,
        llm=llm,
        agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
        memory=memoria,
        agent_kwargs={"prefix": PREFIXO_PTCF, "suffix": SUFIXO,
                      "input_variables": ["input", "chat_history", "agent_scratchpad"]},
        handle_parsing_errors=MSG_ERRO_PARSING,   # erro de formato vira Observation e o agente se corrige
        max_iterations=6,                         # evita laços infinitos (custo controlado)
        early_stopping_method="force",
        return_intermediate_steps=True,           # trajetória disponível para depuração e avaliação
        verbose=False,
    )


def mostrar_trajetoria(res):
    for i, (acao, obs) in enumerate(res["intermediate_steps"], 1):
        pensamento = acao.log.split("Action:")[0].replace("Thought:", "").strip()
        print(f"[{i}] Thought: {pensamento[:260]}")
        print(f"    Action: {acao.tool}  |  Action Input: {acao.tool_input!r}")
        print(f"    Observation: {str(obs)[:230]}...")
    print(f"\n>>> Final Answer: {res['output']}\n")


agente = criar_agente_react(nova_memoria())
print(agente.agent.llm_chain.prompt.template[-900:])

uenos Aires). Use para qualquer pergunta sobre regras, exigências ou dicas. Entrada: a pergunta em linguagem natural.
calculadora(expressao: 'str') -> 'str' - Calculadora aritmética exata. Use para somar custos, dividir orçamentos, porcentagens. Entrada: a expressão, ex.: '2409.97 / 10'.

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [previsao_tempo, converter_moeda, buscar_regras_viagem, calculadora]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Comece! Raciocine no Thought antes de agir.

Histórico da conversa:
{chat_history}

Question: {input}
Thought:{agent_scratchpad}


/var/folders/zd/_w9xft916wnc9s7ywl1sfjm40000gp/T/ipykernel_40774/1128120141.py:15: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  return ConversationSummaryBufferMemory(
/var/folders/zd/_w9xft916wnc9s7ywl1sfjm40000gp/T/ipykernel_40774/1128120141.py:21: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  return initialize_agent(


## 6. O ciclo ReAct em ação

Cinco casos, cada um com memória nova:

1. **Reativo, uma ferramenta:** uma pergunta de clima.
2. **Deliberativo, várias etapas:** orçamento, câmbio, cálculo e regras de entrada.
3. **Ambíguo:** "Porto", tratado conforme a regra do contexto do PTCF.
4. **Limite conhecido de uma ferramenta:** uma moeda que a Frankfurter não cota (a descrição da ferramenta avisa).
5. **Ferramenta com erro em tempo de execução:** uma cidade que não existe (Pasárgada). A API falha, e o agente precisa contornar o erro sem quebrar.

In [9]:
res = criar_agente_react(nova_memoria()).invoke({"input": "Vai chover em Paris amanhã?"}, config=CFG)
mostrar_trajetoria(res)

[1] Thought: Preciso verificar a previsão do tempo para Paris amanhã, que é 2026-11-13. Vou usar a ferramenta de previsão do tempo para isso.
    Action: previsao_tempo  |  Action Input: 'Paris, 2026-11-13'
    Observation: {"cidade": "Paris", "pais": "França", "tipo": "previsao", "fonte": "Open-Meteo", "periodo_consultado": ["2026-11-13", "2026-11-15"], "dias": [{"data": "2026-11-13", "temp_max": 10.8, "temp_min": 4.5, "chuva_mm": 0.0, "condicao": "...

>>> Final Answer: Em Paris, amanhã (13 de novembro), a previsão é de céu nublado e não há expectativa de chuva. As temperaturas estarão entre 4,5 °C e 10,8 °C [Open-Meteo].



In [10]:
res = criar_agente_react(nova_memoria()).invoke(
    {"input": "Tenho 15.000 reais para 10 dias em Lisboa em março. Quanto dá por dia em euros, "
              "e preciso de visto ou de algum documento para entrar?"}, config=CFG)
mostrar_trajetoria(res)

[1] Thought: Para responder à sua pergunta, preciso dividir em duas etapas: (1) calcular quanto você terá por dia em euros e (2) verificar se você precisa de visto ou algum documento para entrar em Lisboa. 

Primeiro, vou converter os 15.000 BRL para EUR usando a cotação d
    Action: converter_moeda  |  Action Input: '15000 BRL EUR'
    Observation: {"valor": 15000.0, "de": "BRL", "para": "EUR", "convertido": 2409.97, "taxa": 0.160665, "data_cotacao": "2026-11-12", "fonte": "Frankfurter (BCE)"}...
[2] Thought: Agora que converti os 15.000 BRL, sei que você terá aproximadamente 2.409,97 euros. Agora, preciso dividir esse valor por 10 dias para saber quanto você terá por dia.
    Action: calculadora  |  Action Input: '2409.97 / 10'
    Observation: 240.997...
[3] Thought: Agora que calculei quanto você terá por dia, que é aproximadamente 240,99 euros, preciso verificar se você precisa de visto ou de algum documento para entrar em Lisboa.
    Action: buscar_regras_viagem  |  Action Input

In [11]:
res = criar_agente_react(nova_memoria()).invoke({"input": "Vai fazer frio no Porto no fim de semana?"}, config=CFG)
mostrar_trajetoria(res)

[1] Thought: "Porto" é ambíguo, mas como estamos falando de um viajante brasileiro, vou assumir que se refere a Porto, Portugal. Preciso verificar a previsão do tempo para o fim de semana, que é sábado (14 de novembro) e domingo (15 de novembro). Vou consultar a previsão p
    Action: previsao_tempo  |  Action Input: 'Porto, 2026-11-14, 2'
    Observation: {"cidade": "Porto", "pais": "Portugal", "tipo": "previsao", "fonte": "Open-Meteo", "periodo_consultado": ["2026-11-14", "2026-11-15"], "dias": [{"data": "2026-11-14", "temp_max": 16.5, "temp_min": 14.3, "chuva_mm": 19.4, "condicao...

>>> Final Answer: No Porto, Portugal, neste fim de semana, a previsão é de temperaturas entre 14,3 °C e 16,5 °C no sábado, com trovoadas e alta probabilidade de chuva. No domingo, as temperaturas serão semelhantes, com mínima de 14,3 °C e máxima de 16,4 °C, também com chuvas e trovoadas [Open-Meteo].



In [12]:
res_erro = criar_agente_react(nova_memoria()).invoke(
    {"input": "Quanto valem 500 reais em pesos argentinos?"}, config=CFG)
mostrar_trajetoria(res_erro)


>>> Final Answer: Infelizmente, não consigo converter reais para pesos argentinos, pois essa moeda não está disponível nas ferramentas que utilizo. Se precisar de ajuda com outra moeda ou qualquer outra informação, estou à disposição!



In [13]:
res_cidade = criar_agente_react(nova_memoria()).invoke(
    {"input": "Vou para Pasárgada amanhã. Vai fazer sol por lá?"}, config=CFG)
mostrar_trajetoria(res_cidade)

[1] Thought: "Pasárgada" pode se referir a um lugar específico, mas não é uma cidade amplamente reconhecida. Vou assumir que se refere a Pasárgada, no Brasil, já que não há outra referência clara. Preciso da previsão do tempo para amanhã.
    Action: previsao_tempo  |  Action Input: 'Pasárgada, 2026-11-13'
    Observation: Erro em previsao_tempo: Cidade 'Pasárgada' não encontrada no geocodificador Open-Meteo....

>>> Final Answer: Não consegui encontrar a cidade "Pasárgada" para verificar a previsão do tempo. Você poderia me informar se se refere a um local específico ou a outra cidade?



### Análise dos casos (execução real com `gpt-4o-mini`)

- **Caso 1:** um único passo Thought → Action → Observation e a resposta. É o comportamento **reativo**, sem planejamento desnecessário.
- **Caso 2:** o agente **decompôs** o pedido sozinho em três etapas encadeadas (converter → dividir → buscar regras), usando a saída de uma como entrada da outra. É o padrão do segundo exemplo few-shot, generalizado para outro caso.
- **Caso 3:** o `Thought` registra a suposição ("Porto é ambíguo... vou assumir Porto, Portugal"), e a resposta a declara, exatamente como pede o CONTEXTO do PTCF. Na primeira versão do prompt, sem o dia da semana, o agente consultou o período errado para "fim de semana". Incluir "Hoje é quinta-feira, ..." no contexto corrigiu isso.
- **Caso 4:** o agente **não** chamou a ferramenta. Ele leu na descrição que a Frankfurter não cobre ARS e explicou a limitação. Isso mostra que a descrição da ferramenta faz parte do prompt e evita chamadas inúteis. Na primeira execução, o modelo escreveu essa resposta sem o prefixo `Final Answer:`, e o `handle_parsing_errors` devolveu a mensagem de correção, com recuperação no passo seguinte. Depois disso, a regra "toda resposta começa com Final Answer" foi incluída no FORMATO.
- **Caso 5:** a API de geocodificação falhou. A `ToolException` virou uma observação legível ("Cidade não encontrada"), e o agente respondeu ao usuário em vez de levantar exceção.

## 7. Memória entre turnos

Uma conversa de quatro turnos em que **os turnos 2 a 4 não repetem o destino**. O agente só acerta se a memória carregar o contexto. O limite de 250 tokens força a sumarização, para mostrar o mecanismo em ação.

In [14]:
memoria_demo = nova_memoria(limite_tokens=250)
agente_conversa = criar_agente_react(memoria_demo)

turnos = [
    "Vou para Roma daqui a uma semana. Como vai estar o tempo lá?",
    "Se eu levar 3000 reais, quanto dá na moeda de lá?",
    "Preciso de seguro-viagem para esse destino?",
    "Obrigada! Me lembra em uma frase o que conversamos?",
]
for i, t in enumerate(turnos, 1):
    r = agente_conversa.invoke({"input": t}, config=CFG)
    ferramentas = [a.tool for a, _ in r["intermediate_steps"]]
    print(f"--- Turno {i}: {t}\n    ferramentas: {ferramentas}\n    resposta: {r['output'][:400]}\n")

--- Turno 1: Vou para Roma daqui a uma semana. Como vai estar o tempo lá?
    ferramentas: ['previsao_tempo']
    resposta: Em Roma, no dia 19 de novembro, a previsão é de temperaturas entre 14,2 °C e 19,1 °C, com pancadas de chuva fracas. Leve um guarda-chuva e vista-se em camadas para se proteger do frio [Open-Meteo].



--- Turno 2: Se eu levar 3000 reais, quanto dá na moeda de lá?
    ferramentas: ['converter_moeda']
    resposta: Se você levar 3000 reais, isso dará cerca de 482 euros na moeda de lá [Frankfurter].



--- Turno 3: Preciso de seguro-viagem para esse destino?
    ferramentas: ['buscar_regras_viagem']
    resposta: Sim, é obrigatório ter seguro-viagem para Roma, com cobertura mínima de 30.000 euros para despesas médicas, hospitalares e repatriação, válido em todo o Espaço Schengen durante toda a viagem. Alguns cartões de crédito oferecem esse seguro como benefício, mas é importante levar a carta de cobertura em inglês [schengen_regras.md].



--- Turno 4: Obrigada! Me lembra em uma frase o que conversamos?
    ferramentas: []
    resposta: Conversamos sobre a previsão do tempo em Roma para o dia 19 de novembro, a conversão de 3000 reais para euros, e a obrigatoriedade de ter seguro-viagem com cobertura mínima de 30.000 euros para despesas médicas durante a sua estadia.



In [15]:
print("Resumo acumulado (turnos antigos):\n", memoria_demo.moving_summary_buffer or "(vazio)")
print("\nMensagens mantidas literalmente:", len(memoria_demo.chat_memory.messages))
print("Tokens no buffer literal:", memoria_demo.llm.get_num_tokens_from_messages(memoria_demo.chat_memory.messages),
      "(limite:", memoria_demo.max_token_limit, ")")

Resumo acumulado (turnos antigos):
 Viajante (Human) irá para Roma em 19 de novembro, com previsão de temperaturas entre 14,2 °C e 19,1 °C e pancadas de chuva fracas. O viajante deve levar um guarda-chuva e vestir-se em camadas para se proteger do frio. O orçamento, número de viajantes e preferências ainda não foram mencionados.

Mensagens mantidas literalmente: 6
Tokens no buffer literal: 220 (limite: 250 )


## 8. Chains sequenciais e paralelas

Um pedido de planejamento completo é processado em várias etapas:

```
pedido em texto
   │  (sequencial) extração estruturada: a chain da Fase 1, reaproveitada sem mudança
   ▼
PedidoViagem (JSON)
   │  (paralelo) três consultas independentes entre si
   ├── clima   → previsao_tempo
   ├── câmbio  → converter_moeda
   └── regras  → buscar_regras_viagem (FAISS)
   ▼
   │  (sequencial) síntese pelo LLM, usando só os dados coletados
   ▼
plano em texto
```

Clima, câmbio e regras **não dependem uns dos outros**, então rodam em paralelo com `RunnableParallel`. A extração precisa vir antes deles, e a síntese depois de todos.

In [16]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough
from viagens.apis import MOEDA_POR_CIDADE
from viagens.extracao import criar_chain_extracao

extrair = criar_chain_extracao(llm)


def etapa_clima(p):
    if not p.get("destino"):
        return "sem destino informado"
    return por_nome["previsao_tempo"].invoke({"cidade": p["destino"], "data_inicio": p.get("data_ida"), "dias": 3})


def etapa_cambio(p):
    if not p.get("orcamento"):
        return "sem orçamento informado"
    para = MOEDA_POR_CIDADE.get(p.get("destino"), "EUR")
    return por_nome["converter_moeda"].invoke({"valor": p["orcamento"], "de": p.get("moeda_orcamento") or "BRL", "para": para})


def etapa_regras(p):
    return por_nome["buscar_regras_viagem"].invoke({"consulta": f"regras de entrada, visto e documentos para {p.get('destino')}"})


coleta_paralela = RunnableParallel(clima=RunnableLambda(etapa_clima),
                                   cambio=RunnableLambda(etapa_cambio),
                                   regras=RunnableLambda(etapa_regras))

prompt_sintese = ChatPromptTemplate.from_messages([
    ("system", "Você vai redigir a síntese do plano de viagem para o usuário, em português, em até 8 linhas, "
               "usando SOMENTE os dados abaixo e citando a fonte de cada dado entre colchetes."),
    ("human", "Pedido estruturado: {pedido}\n\nCLIMA: {clima}\n\nCAMBIO: {cambio}\n\nREGRAS: {regras}"),
])

pipeline_planejamento = (
    RunnableParallel(pedido=extrair)                                          # 1. sequencial
    | RunnablePassthrough.assign(dados=RunnableLambda(lambda x: x["pedido"]) | coleta_paralela)  # 2. paralelo
    | RunnableLambda(lambda x: {"pedido": json.dumps(x["pedido"], ensure_ascii=False),
                                **{k: str(v)[:1500] for k, v in x["dados"].items()}})
    | prompt_sintese | llm | StrOutputParser()                               # 3. sequencial
).with_config(run_name="pipeline_planejamento")

pedido_txt = "Quero planejar 7 dias em Barcelona daqui a 10 dias, somos 2 pessoas e temos 12 mil reais."
print(pipeline_planejamento.invoke({"pedido": pedido_txt}, config=CFG))

Você planejou uma viagem para Barcelona, Espanha, de 22 a 29 de novembro de 2026, para duas pessoas, com um orçamento de R$ 12.000,00 (aproximadamente € 1.927,99) [Frankfurter (BCE)]. O clima durante sua estadia deve ser ameno, com temperaturas variando entre 6,5 °C e 18,5 °C, e possibilidade de garoa nos primeiros dias [Open-Meteo]. Para a locomoção, considere o cartão T-casual para 10 viagens em metrô e ônibus [guia_barcelona.md]. Lembre-se de que é necessário um passaporte válido, passagem de volta, comprovante de hospedagem e seguro-viagem [portugal_entrada.md].


In [17]:
# Ganho do paralelismo: as mesmas três etapas, em sequência e em paralelo
p = extrair.invoke({"pedido": pedido_txt})

t0 = time.perf_counter()
_ = [etapa_clima(p), etapa_cambio(p), etapa_regras(p)]
t_seq = time.perf_counter() - t0

t0 = time.perf_counter()
_ = coleta_paralela.invoke(p)
t_par = time.perf_counter() - t0
print(f"sequencial: {t_seq:.2f}s | paralelo: {t_par:.2f}s | ganho: {t_seq / max(t_par, 1e-6):.1f}x")

sequencial: 1.83s | paralelo: 0.47s | ganho: 3.9x


**Observação para as próximas fases.** Nas execuções reais, a síntese de Barcelona atribuiu a exigência de passaporte e seguro a um documento que **não contém** essa regra: `portugal_entrada.md` em uma execução, `guia_barcelona.md` em outra. A regra vale para todo o Schengen e está em `schengen_regras.md`, que não foi recuperado. A causa está na recuperação: a consulta usa a **cidade** ("Barcelona"), enquanto os documentos de regras falam em **países** ("Espanha", "Espaço Schengen"). É um descompasso de vocabulário entre a pergunta e o texto indexado, que será diagnosticado e corrigido com transformação de consulta na Fase 4. A citação de fonte inadequada é o tipo de erro que a verificação por NLI da Fase 5 deve sinalizar.

## 9. Roteamento dinâmico com `RouterChain`

Nem toda mensagem precisa do pipeline completo. O `LLMRouterChain` lê a pergunta e escolhe **em tempo de execução** o ramo adequado. O `MultiRouteChain` executa o ramo escolhido.

| Ramo | Quando | O que executa |
|---|---|---|
| `clima` | tempo, temperatura, chuva | extração → `previsao_tempo` → resposta |
| `cambio` | cotação, conversão | *function calling* com `converter_moeda` |
| `regras` | visto, bagagem, documentos | FAISS → resposta baseada no contexto (RAG) |
| `planejamento` | planejar uma viagem completa | o pipeline sequencial + paralelo da seção 8 |
| `DEFAULT` | fora do domínio | LLM direto, com aviso de escopo |

In [18]:
from langchain.chains import TransformChain
from langchain.chains.router.base import MultiRouteChain
from langchain.chains.router.llm_router import LLMRouterChain, RouterOutputParser
from langchain.chains.router.multi_prompt_prompt import MULTI_PROMPT_ROUTER_TEMPLATE
from langchain_core.prompts import PromptTemplate

prompt_rag = ChatPromptTemplate.from_messages([
    ("system", "Você é um assistente de viagens. Responda usando apenas o contexto abaixo; se ele não "
               "trouxer a resposta, diga que não sabe. Cite a fonte entre colchetes."),
    ("human", "Contexto:\n{contexto}\n\nPergunta: {pergunta}"),
])
responder_com_contexto = prompt_rag | llm | StrOutputParser()

# Para dados de API o prompt é outro: o contexto é um JSON atual que precisa ser INTERPRETADO.
# (Na 1ª execução, usar o prompt de RAG aqui fez o modelo responder "Não sei." para o câmbio.)
prompt_dados = ChatPromptTemplate.from_messages([
    ("system", "Você é um assistente de viagens. Responda usando apenas o contexto abaixo: ele contém dados "
               "em JSON devolvidos agora por uma API em tempo real (as datas no JSON são as da consulta). "
               "Interprete os campos e responda em até 3 frases, citando a fonte entre colchetes."),
    ("human", "Contexto:\n{contexto}\n\nPergunta: {pergunta}"),
])
responder_com_dados = prompt_dados | llm | StrOutputParser()


def ramo_clima(texto):
    dados = etapa_clima(extrair.invoke({"pedido": texto}))
    return responder_com_dados.invoke({"contexto": dados, "pergunta": texto})


def ramo_cambio(texto):
    ai = llm.bind_tools([por_nome["converter_moeda"]]).invoke(texto)
    if not ai.tool_calls:
        return ai.content
    dados = por_nome["converter_moeda"].invoke(ai.tool_calls[0]["args"])
    return responder_com_dados.invoke({"contexto": dados, "pergunta": texto})


def ramo_regras(texto):
    trechos = json.loads(por_nome["buscar_regras_viagem"].invoke({"consulta": texto}))
    contexto = "\n\n".join(f"[base: {t['fonte']}] {t['trecho']}" for t in trechos)
    return responder_com_contexto.invoke({"contexto": contexto, "pergunta": texto})


def ramo_planejamento(texto):
    return pipeline_planejamento.invoke({"pedido": texto})


def ramo_default(texto):
    return (ChatPromptTemplate.from_template(
        "Você é um assistente de VIAGENS. Se a pergunta estiver fora desse escopo, diga isso em uma frase "
        "e ofereça ajuda com viagens. Pergunta: {q}") | llm | StrOutputParser()).invoke({"q": texto})


def como_chain(fn, nome):
    return TransformChain(input_variables=["input"], output_variables=["text"],
                          transform=lambda d: {"text": fn(d["input"])}, name=nome)


destinos = {
    "clima": "perguntas sobre previsão do tempo, temperatura, chuva ou clima de uma cidade",
    "cambio": "perguntas sobre cotação de moedas ou conversão de valores entre moedas",
    "regras": "perguntas sobre visto, documentos, regras de entrada, seguro, bagagem ou o que levar",
    "planejamento": "pedidos para planejar uma viagem completa (destino com datas, duração e/ou orçamento)",
}
router_prompt = PromptTemplate(
    template=MULTI_PROMPT_ROUTER_TEMPLATE.format(destinations="\n".join(f"{k}: {v}" for k, v in destinos.items())),
    input_variables=["input"], output_parser=RouterOutputParser())
router_chain = LLMRouterChain.from_llm(llm, router_prompt)

roteador = MultiRouteChain(
    router_chain=router_chain,
    destination_chains={"clima": como_chain(ramo_clima, "ramo_clima"),
                        "cambio": como_chain(ramo_cambio, "ramo_cambio"),
                        "regras": como_chain(ramo_regras, "ramo_regras"),
                        "planejamento": como_chain(ramo_planejamento, "ramo_planejamento")},
    default_chain=como_chain(ramo_default, "ramo_default"),
    silent_errors=True,   # destino desconhecido cai no DEFAULT em vez de levantar exceção
)

In [19]:
perguntas = [
    "Vai chover em Barcelona amanhã?",
    "Quanto dá 300 dólares em reais hoje?",
    "Posso levar power bank na mala despachada?",
    "Planeje uma viagem de 7 dias para Paris em maio com 12 mil reais",
    "Qual a capital da Austrália?",
]
for q in perguntas:
    rota = router_chain.route({"input": q}, callbacks=[log_handler])
    resp = roteador.invoke({"input": q}, config=CFG)["text"]
    print(f"[{rota.destination or 'DEFAULT':>12}] {q}\n               → {' '.join(resp.split())[:300]}\n")

[       clima] Vai chover em Barcelona amanhã?
               → A previsão para Barcelona no dia 14 de novembro de 2026 indica a possibilidade de pancadas de chuva fracas, com uma probabilidade de 20% e uma expectativa de 0,5 mm de chuva. Portanto, há uma chance de chuva, mas não é garantido. [Open-Meteo]



[      cambio] Quanto dá 300 dólares em reais hoje?
               → Hoje, 300 dólares equivalem a aproximadamente 1685,07 reais, com uma taxa de câmbio de 5,6169. Essa cotação foi registrada em 12 de novembro de 2026 [Frankfurter (BCE)].



[      regras] Posso levar power bank na mala despachada?
               → Não, baterias de lítio e power banks só podem ir na bagagem de mão, nunca na despachada [base: bagagem_aerosul.md].



[planejamento] Planeje uma viagem de 7 dias para Paris em maio com 12 mil reais
               → Você está planejando uma viagem a Paris de 1 a 8 de maio de 2027, com um orçamento de R$ 12.000,00 (aproximadamente € 1.927,99) [Frankfurter (BCE)]. O clima nessa época costuma ser ameno, com temperaturas variando entre 11,7°C e 25,5°C, podendo ocorrer chuvas leves [Open-Meteo]. Para a entrada na Fr



[     DEFAULT] Qual a capital da Austrália?
               → A capital da Austrália é Canberra. Se precisar de ajuda com viagens, estou à disposição!



## 10. Output parsers estruturados com tratamento de erros

A resposta final do agente é texto. Para que o resto do sistema (interface, verificação na Fase 5, avaliação) possa confiar nela, ela é convertida em um objeto `RespostaViagem`, em **três camadas de defesa**:

1. **`PydanticOutputParser`**: valida tipos e campos obrigatórios. Se falhar, levanta `OutputParserException` (não deixa passar lixo em silêncio).
2. **`OutputFixingParser`**: ao falhar, devolve ao LLM a saída inválida e o erro, e pede uma correção (até 2 tentativas).
3. **`with_fallbacks`**: se mesmo assim falhar (API fora do ar, resposta irrecuperável), devolve um objeto válido com `confianca="baixa"` e o texto bruto. O usuário sempre recebe uma resposta, e o sistema sabe que ela é de baixa confiança.

Além disso, no nível do agente, o `handle_parsing_errors` trata erros no formato ReAct (seção 5).

In [20]:
from typing import List, Literal
from pydantic import BaseModel, Field
from langchain.output_parsers import OutputFixingParser, PydanticOutputParser
from langchain_core.exceptions import OutputParserException


class RespostaViagem(BaseModel):
    resposta: str = Field(description="Resposta ao usuário, em português")
    fontes: List[str] = Field(description="Ferramentas ou documentos usados (ex.: Open-Meteo, Frankfurter, guia_lisboa.md)")
    confianca: Literal["alta", "media", "baixa"] = Field(description="Confiança na resposta")
    precisa_mais_info: bool = Field(description="True se faltou informação do usuário para responder bem")


parser_resp = PydanticOutputParser(pydantic_object=RespostaViagem)

# Saída típica de um LLM "quase certo": dicionário Python (aspas simples, vírgula sobrando, False)
saida_invalida = ("{'resposta': 'Em Lisboa a previsão é de 16 a 20 °C.', 'fontes': ['Open-Meteo'], "
                  "'confianca': 'alta', 'precisa_mais_info': False,}")
try:
    parser_resp.parse(saida_invalida)
except OutputParserException as e:
    print("Camada 1, PydanticOutputParser rejeitou:\n ", str(e).splitlines()[0][:160])

parser_corretor = OutputFixingParser.from_llm(parser=parser_resp, llm=llm, max_retries=2)
print("\nCamada 2, OutputFixingParser corrigiu:\n ", parser_corretor.parse(saida_invalida))

Camada 1, PydanticOutputParser rejeitou:
  Invalid json output: {'resposta': 'Em Lisboa a previsão é de 16 a 20 °C.', 'fontes': ['Open-Meteo'], 'confianca': 'alta', 'precisa_mais_info': False,}



Camada 2, OutputFixingParser corrigiu:
  resposta='Em Lisboa a previsão é de 16 a 20 °C.' fontes=['Open-Meteo'] confianca='alta' precisa_mais_info=False


In [21]:
prompt_formatar = ChatPromptTemplate.from_messages([
    ("system", "Converta a resposta do agente de viagens para o formato RespostaViagem. Não altere os fatos. "
               "Liste em 'fontes' as ferramentas ou documentos citados."),
    ("human", "Texto do agente:\n{texto}\n\n### Formato\n{format_instructions}"),
]).partial(format_instructions=parser_resp.get_format_instructions())


def resposta_de_baixa_confianca(entrada: dict) -> RespostaViagem:
    return RespostaViagem(resposta=entrada["texto"], fontes=[], confianca="baixa", precisa_mais_info=True)


formatar_resposta = (prompt_formatar | llm | parser_corretor).with_fallbacks(
    [RunnableLambda(resposta_de_baixa_confianca)]).with_config(run_name="formatar_resposta")

print("Resposta do agente (seção 6, caso do peso argentino) estruturada:")
print(formatar_resposta.invoke({"texto": res_erro["output"]}, config=CFG).model_dump_json(indent=2))

# Camada 3: simulamos um modelo que devolve algo irrecuperável -> o fallback garante uma saída válida
modelo_quebrado = RunnableLambda(lambda _: "desculpe, não consigo responder em JSON")
formatar_quebrado = (prompt_formatar | modelo_quebrado | parser_resp).with_fallbacks(
    [RunnableLambda(resposta_de_baixa_confianca)])
print("\nCamada 3, fallback:", formatar_quebrado.invoke({"texto": "Texto qualquer do agente."}))

Resposta do agente (seção 6, caso do peso argentino) estruturada:


{
  "resposta": "Infelizmente, não consigo converter reais para pesos argentinos, pois essa moeda não está disponível nas ferramentas que utilizo. Se precisar de ajuda com outra moeda ou qualquer outra informação, estou à disposição!",
  "fontes": [],
  "confianca": "alta",
  "precisa_mais_info": false
}

Camada 3, fallback: resposta='Texto qualquer do agente.' fontes=[] confianca='baixa' precisa_mais_info=True


In [22]:
# Erro de formato no próprio ciclo ReAct: o parser do agente rejeita e o executor devolve a
# mensagem MSG_ERRO_PARSING como Observation, para o modelo se corrigir no passo seguinte.
from langchain.agents.mrkl.output_parser import MRKLOutputParser
try:
    MRKLOutputParser().parse("Acho que vou olhar o clima de Lisboa primeiro.")
except OutputParserException as e:
    print("O parser ReAct rejeitou:", str(e)[:120])
    print("O executor devolve ao modelo:", MSG_ERRO_PARSING)

O parser ReAct rejeitou: Could not parse LLM output: `Acho que vou olhar o clima de Lisboa primeiro.`
For troubleshooting, visit: https://python.
O executor devolve ao modelo: Formato inválido. Responda com 'Thought:' seguido de 'Action:' e 'Action Input:' OU de 'Final Answer:'. Não escreva os dois ao mesmo tempo.


## 11. Callbacks para logging e depuração

O `LogExecucaoHandler` (`viagens/callbacks.py`) foi passado em `config={"callbacks": [...]}` em todas as execuções acima. Passado assim, por requisição, ele se propaga para **todos** os sub-runs: LLM, ferramentas, agente e parsers. Se fosse passado no construtor, só a chain externa seria observada. Cada evento vai para `logs/agente.log` (um JSON por linha) e para uma tabela em memória.

In [23]:
eventos = log_handler.resumo()
print(f"{len(eventos)} eventos registrados\n")
print(eventos["evento"].value_counts().to_string(), "\n")

tools_fim = eventos[eventos["evento"] == "tool_fim"]
acoes = eventos[eventos["evento"] == "agente_acao"]
print("Ações do agente ReAct por ferramenta:\n", acoes["ferramenta"].value_counts().to_string(), "\n")
print("Latência das ferramentas (s): média", round(tools_fim["duracao_s"].mean(), 3),
      "| máx", round(tools_fim["duracao_s"].max(), 3))
print("Erros de ferramenta tratados:", int(tools_fim["erro_tratado"].eq(True).sum()))
if "tokens" in eventos and eventos["tokens"].notna().any():
    print("Tokens consumidos (LLM):", int(eventos["tokens"].fillna(0).sum()))

87 eventos registrados

evento
llm_fim        33
tool_inicio    18
tool_fim       18
agente_acao     9
agente_fim      9 

Ações do agente ReAct por ferramenta:
 ferramenta
previsao_tempo          4
converter_moeda         2
buscar_regras_viagem    2
calculadora             1 

Latência das ferramentas (s): média 0.432 | máx 0.696
Erros de ferramenta tratados: 2
Tokens consumidos (LLM): 35907


In [24]:
# As últimas linhas do arquivo de log (formato JSON por linha, fácil de filtrar com jq/pandas)
linhas = (ROOT / "logs" / "agente.log").read_text(encoding="utf-8").strip().splitlines()
for l in linhas[-4:]:
    print(l[:230])

{"ts": "20:55:47", "sessao": "fase2", "evento": "llm_fim", "duracao_s": 0.923, "tokens": 360}
{"ts": "20:55:56", "sessao": "fase2", "evento": "llm_fim", "duracao_s": 0.936, "tokens": 338}
{"ts": "20:55:57", "sessao": "fase2", "evento": "llm_fim", "duracao_s": 1.194, "tokens": 338}
{"ts": "20:56:01", "sessao": "fase2", "evento": "llm_fim", "duracao_s": 1.836, "tokens": 474}


In [25]:
if not USE_DUMMY:
    print("Traces completos no LangSmith → projeto:", os.getenv("LANGSMITH_PROJECT"),
          "| endpoint:", os.getenv("LANGSMITH_ENDPOINT"))
else:
    print("Modo dummy: tracing do LangSmith desligado; o log local acima substitui o trace.")

Traces completos no LangSmith → projeto: assistente-viagens | endpoint: https://eu.api.smith.langchain.com


## 12. Justificativa das escolhas de confiabilidade

| Componente | Escolha | Por que esta escolha (e não outra) |
|---|---|---|
| **Formato de ferramenta para o ReAct** | `Tool` de entrada única (string) | O `ZERO_SHOT_REACT_DESCRIPTION` só aceita ferramentas de uma entrada. As versões estruturadas (Pydantic) ficam para o *function calling* e para o agente final, em que os argumentos tipados eliminam a etapa de parsing de texto. |
| **Erros de ferramenta** | `ToolException` + `handle_tool_error=True` | Uma falha de API (moeda não cotada, cidade inexistente, timeout) vira **observação** que o agente lê e contorna (seção 6, caso 4), em vez de uma exceção que mata a conversa. |
| **Erros de formato do ReAct** | `handle_parsing_errors` com mensagem específica | Uma mensagem que diz *como* corrigir é mais eficaz que o padrão genérico, e o modelo se recupera no passo seguinte. |
| **Limites do ciclo** | `max_iterations=6`, `early_stopping_method="force"` | O caso mais longo do domínio (planejamento completo) usa de 3 a 4 ferramentas. Seis passos dão folga sem permitir laços caros. |
| **Memória** | `ConversationSummaryBufferMemory(max_token_limit=800)` | Mantém literal o que é recente e resume o que é antigo, preservando preferências do início da conversa com custo limitado (seção 5). |
| **Saída final** | `PydanticOutputParser` → `OutputFixingParser` → `with_fallbacks` | Validação estrita, correção automática barata e degradação graciosa, nessa ordem. O sistema nunca devolve algo fora do contrato e marca a baixa confiança quando degrada. |
| **Busca semântica** | FAISS local, índice persistido em disco | Base pequena (31 chunks): o FAISS em memória responde em milissegundos, não exige servidor e roda no Colab. Persistir evita reindexar (e pagar embeddings) a cada execução. |
| **Observabilidade** | Callback local + LangSmith | O callback funciona offline e dá métricas imediatas (latência, erros, contagem de ferramentas); o LangSmith guarda o trace completo para a avaliação de trajetória da Fase 5. |
| **Temperatura** | 0 nas decisões | A escolha de ferramenta e a extração precisam ser reprodutíveis para depurar e avaliar. |

## 13. Resultado da fase e próximos passos

**Entregue nesta fase**
- A comparação de orquestradores e a justificativa do LangChain.
- Quatro ferramentas reais (duas APIs externas, FAISS e calculadora), com a API externa funcionando via **function calling**.
- O prompt PTCF com few-shot e chain-of-thought voltado aos casos ambíguos do domínio.
- O agente ReAct (`initialize_agent` + `ZERO_SHOT_REACT_DESCRIPTION`) resolvendo casos de uma etapa, de várias etapas, ambíguos e com erro de ferramenta.
- A memória com sumarização mantendo o contexto entre turnos.
- Chains sequenciais e paralelas, com o ganho do paralelismo medido, e o roteamento dinâmico com `RouterChain`.
- Os parsers com três camadas de tratamento de erro e os callbacks de log.

**Próxima fase (3):** a estratégia de memória de curto e longo prazo pela taxonomia CoALA e o chatbot com `ConversationBufferMemory`, reconhecimento de intenção e extração de entidades.